
# Lantern candidate analysis — hybrid multi-loci v8, no Gaia color cut

This notebook (previously named `lantern_candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut`) is a **minimal variant of the v8 candidate-analysis workflow**.

The Gaia cleaning is changed to:

- **keep** the proper-motion cut;
- **keep** the parallax-significance cut;
- **do not reject** candidates because of Gaia BP−RP color;
- separately identify and save candidates with **Gaia BP−RP >= 2.5** after the PM/parallax cuts.

The important selection is therefore

```python
keep_flag = ~(high_pm_flag | high_parallax_flag)
```

and **not**

```python
keep_flag = ~(high_pm_flag | high_parallax_flag | very_red_flag)
```

`very_red_flag` is retained only as a diagnostic for compatibility with the original v8 logic.

## New output folder

```text
candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut/
```

## Main outputs

```text
second_pass_candidates_with_flags.parquet
second_pass_candidates_with_flags.csv

second_pass_candidates_filtered.parquet
second_pass_candidates_filtered.csv

second_pass_candidates_gaia_bp_rp_ge_2p5.parquet
second_pass_candidates_gaia_bp_rp_ge_2p5.csv
```

The last pair contains candidates that:

1. pass the Gaia PM cut;
2. pass the Gaia parallax cut;
3. have `gaia_bp_rp_max >= 2.5`.


In [ ]:

from pathlib import Path
import ast
import json
from collections.abc import Mapping

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt


# ============================================================
# INPUTS — same generation of products used by v8
# ============================================================

SECOND_PASS_DIR = Path(
    "second_pass_output_hybrid_multi_loci"
)

CANDIDATES_FILE = (
    SECOND_PASS_DIR
    / "second_pass_candidates.parquet"
)

CANDIDATE_ALERTS_FILE = (
    SECOND_PASS_DIR
    / "second_pass_candidate_alerts.parquet"
)

DATA_DIR = Path(
    "../download_v4/antares_data_hybrid"
)

LOCI_DIR = (
    DATA_DIR
    / "loci"
)


# ============================================================
# NEW OUTPUT FOLDER
# ============================================================

OUTPUT_DIR = Path(
    "candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OVERWRITE_OUTPUT = True


# ============================================================
# GAIA SETTINGS
# ============================================================

PM_SIG_CUT = 3.0
PARALLAX_SIG_CUT = 3.0

# This is now only used to DEFINE the red-Gaia subset.
# It is NOT used to reject candidates.
GAIA_BP_RP_THRESHOLD = 2.5


# ============================================================
# PHOTOMETRY
# ============================================================

AB_ZEROPOINT_NJY = 31.4
BANDS = "ugrizy"


# ============================================================
# PLOTTING
# ============================================================

HEXBIN_THRESHOLD = 1500
HEXBIN_GRIDSIZE = 100

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})


## 1. Load Second-Pass candidates and candidate alerts

In [ ]:

candidates = pd.read_parquet(
    CANDIDATES_FILE
)

candidate_alerts = pd.read_parquet(
    CANDIDATE_ALERTS_FILE
)


required_candidates = {
    "target_id",
    "representative_locus_id",
    "member_locus_ids",
    "n_member_loci",
    "ra",
    "dec",
    "first_cut_max_score",
    "n_tagged",
    "n_lsst_alerts",
    "n_detections",
    "percent_tagged",
    "second_pass_score",
}

required_alerts = {
    "target_id",
    "locus_id",
    "alert_id",
    "mjd",
    "alert_properties",
}


missing = (
    required_candidates
    - set(candidates.columns)
)

if missing:
    raise KeyError(
        "Candidate table is missing required columns: "
        f"{sorted(missing)}"
    )


missing = (
    required_alerts
    - set(candidate_alerts.columns)
)

if missing:
    raise KeyError(
        "Candidate-alert table is missing required columns: "
        f"{sorted(missing)}"
    )


if candidates["target_id"].duplicated().any():
    raise ValueError(
        "Duplicate target_id rows in candidate table."
    )


if candidate_alerts["alert_id"].duplicated().any():
    raise ValueError(
        "Duplicate alert_id rows in candidate-alert table."
    )


print(
    f"Candidate targets: {len(candidates):,}"
)

print(
    f"Candidate alerts:  {len(candidate_alerts):,}"
)

print(
    "Multi-locus candidates: "
    f"{(candidates['n_member_loci'] > 1).sum():,}"
)


## 2. Helper functions

In [ ]:

def parse_value(value):
    if isinstance(value, Mapping):
        return dict(value)

    if isinstance(value, str):
        for parser in (
            json.loads,
            ast.literal_eval,
        ):
            try:
                return parser(value)
            except Exception:
                pass

    return value


def parse_properties(value):
    parsed = parse_value(
        value
    )

    if isinstance(parsed, Mapping):
        return dict(parsed)

    return {}


def as_list(value):
    value = parse_value(
        value
    )

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return list(value)

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def to_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def to_bool(value):
    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(value)

    if isinstance(value, str):
        x = value.strip().lower()

        if x == "true":
            return True

        if x == "false":
            return False

    try:
        if pd.isna(value):
            return False
    except Exception:
        pass

    return bool(value)


def flux_njy_to_abmag(flux):
    flux = to_float(
        flux
    )

    if (
        not np.isfinite(flux)
        or flux <= 0
    ):
        return np.nan

    return (
        AB_ZEROPOINT_NJY
        - 2.5
        * np.log10(
            flux
        )
    )


def max_finite(values):
    values = [
        x
        for x in values
        if np.isfinite(x)
    ]

    return (
        float(np.max(values))
        if values
        else np.nan
    )


def min_finite(values):
    values = [
        x
        for x in values
        if np.isfinite(x)
    ]

    return (
        float(np.min(values))
        if values
        else np.nan
    )


## 3. Load raw locus metadata used by the original v8 Gaia cleaning

In [ ]:

member_locus_ids = []

for value in candidates[
    "member_locus_ids"
]:
    member_locus_ids.extend(
        [
            str(x)
            for x
            in as_list(
                value
            )
        ]
    )

member_locus_ids = sorted(
    set(
        member_locus_ids
    )
)


locus_columns = [
    "locus_id",
    "ra",
    "dec",
    "tags",
    "catalogs",
    "num_tagged_alerts",
    "max_score",

    "gaia_parallax_over_error",
    "gaia_pmdec",
    "gaia_pmdec_error",
    "gaia_pmra",
    "gaia_pmra_error",
    "gaia_in_qso_candidates",
    "gaia_phot_g_mean_mag",
    "gaia_phot_g_mean_flux_over_error",
    "gaia_bp_g",
    "gaia_g_rp",

    "milliquas_match",
]


loci_dataset = ds.dataset(
    LOCI_DIR,
    format="parquet",
)


available_locus_columns = set(
    loci_dataset.schema.names
)


load_locus_columns = [
    col
    for col
    in locus_columns
    if col
    in available_locus_columns
]


required_locus_columns = {
    "locus_id",
}

missing = (
    required_locus_columns
    - set(
        load_locus_columns
    )
)

if missing:
    raise KeyError(
        "Raw locus dataset is missing required columns: "
        f"{sorted(missing)}"
    )


try:
    loci = (
        loci_dataset
        .to_table(
            columns=load_locus_columns,
            filter=ds.field(
                "locus_id"
            ).isin(
                member_locus_ids
            ),
        )
        .to_pandas()
    )

except Exception as e:
    print(
        "Filtered locus read failed; loading requested columns "
        f"and filtering afterward. Reason: {e}"
    )

    loci = (
        loci_dataset
        .to_table(
            columns=load_locus_columns
        )
        .to_pandas()
    )

    loci = loci[
        loci[
            "locus_id"
        ]
        .astype(str)
        .isin(
            member_locus_ids
        )
    ].copy()


if loci[
    "locus_id"
].duplicated().any():

    print(
        "Warning: duplicate locus_id rows found; "
        "keeping the first row."
    )

    loci = loci.drop_duplicates(
        "locus_id",
        keep="first",
    )


print(
    f"Relevant member loci loaded: {len(loci):,}"
)


## 4. Build per-locus Gaia summaries

In [ ]:

def summarize_locus_gaia(row):
    # --------------------------------------------------------
    # Gaia arrays
    # --------------------------------------------------------

    pmra = as_list(
        row.get(
            "gaia_pmra"
        )
    )

    pmra_err = as_list(
        row.get(
            "gaia_pmra_error"
        )
    )

    pmdec = as_list(
        row.get(
            "gaia_pmdec"
        )
    )

    pmdec_err = as_list(
        row.get(
            "gaia_pmdec_error"
        )
    )

    parallax_over_error = as_list(
        row.get(
            "gaia_parallax_over_error"
        )
    )

    bp_g = as_list(
        row.get(
            "gaia_bp_g"
        )
    )

    g_rp = as_list(
        row.get(
            "gaia_g_rp"
        )
    )

    g_mag = as_list(
        row.get(
            "gaia_phot_g_mean_mag"
        )
    )

    gaia_qso = as_list(
        row.get(
            "gaia_in_qso_candidates"
        )
    )


    n_gaia = max(
        len(pmra),
        len(pmra_err),
        len(pmdec),
        len(pmdec_err),
        len(parallax_over_error),
        len(bp_g),
        len(g_rp),
        len(g_mag),
        len(gaia_qso),
        0,
    )


    # --------------------------------------------------------
    # Proper-motion significance
    # sqrt[(pmra/err)^2 + (pmdec/err)^2]
    # --------------------------------------------------------

    pm_sig = []

    for i in range(
        n_gaia
    ):
        a = (
            to_float(
                pmra[i]
            )
            if i < len(pmra)
            else np.nan
        )

        ae = (
            to_float(
                pmra_err[i]
            )
            if i < len(pmra_err)
            else np.nan
        )

        d = (
            to_float(
                pmdec[i]
            )
            if i < len(pmdec)
            else np.nan
        )

        de = (
            to_float(
                pmdec_err[i]
            )
            if i < len(pmdec_err)
            else np.nan
        )

        if (
            np.isfinite(a)
            and np.isfinite(ae)
            and ae > 0
            and np.isfinite(d)
            and np.isfinite(de)
            and de > 0
        ):
            pm_sig.append(
                np.sqrt(
                    (a / ae)**2
                    + (d / de)**2
                )
            )


    # --------------------------------------------------------
    # |parallax / error|
    # --------------------------------------------------------

    parallax_sig = []

    for value in parallax_over_error:
        x = to_float(
            value
        )

        if np.isfinite(
            x
        ):
            parallax_sig.append(
                abs(
                    x
                )
            )


    # --------------------------------------------------------
    # Gaia BP-RP = (BP-G) + (G-RP)
    # --------------------------------------------------------

    bp_rp = []

    for i in range(
        max(
            len(bp_g),
            len(g_rp),
        )
    ):
        a = (
            to_float(
                bp_g[i]
            )
            if i < len(bp_g)
            else np.nan
        )

        b = (
            to_float(
                g_rp[i]
            )
            if i < len(g_rp)
            else np.nan
        )

        if (
            np.isfinite(a)
            and np.isfinite(b)
        ):
            bp_rp.append(
                a + b
            )


    g_mag_finite = [
        to_float(x)
        for x in g_mag
        if np.isfinite(
            to_float(
                x
            )
        )
    ]


    qso_any = any(
        to_bool(
            value
        )
        for value
        in gaia_qso
    )


    milliquas_match = (
        to_bool(
            row.get(
                "milliquas_match",
                False,
            )
        )
    )


    return pd.Series(
        {
            "locus_id":
                str(
                    row[
                        "locus_id"
                    ]
                ),

            "gaia_n_matches_locus":
                int(
                    n_gaia
                ),

            "gaia_pm_sig_max_locus":
                max_finite(
                    pm_sig
                ),

            "gaia_parallax_sig_max_locus":
                max_finite(
                    parallax_sig
                ),

            "gaia_bp_rp_max_locus":
                max_finite(
                    bp_rp
                ),

            "gaia_phot_g_mean_mag_brightest_locus":
                min_finite(
                    g_mag_finite
                ),

            "gaia_in_qso_candidates_any_locus":
                bool(
                    qso_any
                ),

            "milliquas_match_locus":
                bool(
                    milliquas_match
                ),
        }
    )


locus_gaia = loci.apply(
    summarize_locus_gaia,
    axis=1,
)


display(
    locus_gaia.head()
)


## 5. Aggregate Gaia summaries across all member loci of each target

In [ ]:

locus_gaia_indexed = locus_gaia.set_index(
    "locus_id"
)


target_rows = []


for _, candidate in candidates.iterrows():

    target_id = candidate[
        "target_id"
    ]

    member_ids = [
        str(x)
        for x
        in as_list(
            candidate[
                "member_locus_ids"
            ]
        )
    ]


    rows = locus_gaia_indexed.reindex(
        member_ids
    )


    n_matches = pd.to_numeric(
        rows[
            "gaia_n_matches_locus"
        ],
        errors="coerce",
    ).fillna(
        0
    )


    pm = pd.to_numeric(
        rows[
            "gaia_pm_sig_max_locus"
        ],
        errors="coerce",
    )


    parallax = pd.to_numeric(
        rows[
            "gaia_parallax_sig_max_locus"
        ],
        errors="coerce",
    )


    color = pd.to_numeric(
        rows[
            "gaia_bp_rp_max_locus"
        ],
        errors="coerce",
    )


    gmag = pd.to_numeric(
        rows[
            "gaia_phot_g_mean_mag_brightest_locus"
        ],
        errors="coerce",
    )


    target_rows.append(
        {
            "target_id":
                target_id,

            "gaia_match":
                bool(
                    n_matches.sum()
                    > 0
                ),

            "gaia_n_matches":
                int(
                    n_matches.sum()
                ),

            "gaia_pm_sig_max":
                (
                    pm.max()
                    if pm.notna().any()
                    else np.nan
                ),

            "gaia_parallax_sig_max":
                (
                    parallax.max()
                    if parallax.notna().any()
                    else np.nan
                ),

            "gaia_bp_rp_max":
                (
                    color.max()
                    if color.notna().any()
                    else np.nan
                ),

            "gaia_phot_g_mean_mag_brightest":
                (
                    gmag.min()
                    if gmag.notna().any()
                    else np.nan
                ),

            "gaia_in_qso_candidates_any":
                bool(
                    rows[
                        "gaia_in_qso_candidates_any_locus"
                    ]
                    .fillna(
                        False
                    )
                    .astype(
                        bool
                    )
                    .any()
                ),

            "milliquas_match":
                bool(
                    rows[
                        "milliquas_match_locus"
                    ]
                    .fillna(
                        False
                    )
                    .astype(
                        bool
                    )
                    .any()
                ),
        }
    )


target_gaia = pd.DataFrame(
    target_rows
)


analysis = candidates.merge(
    target_gaia,
    on="target_id",
    how="left",
    validate="one_to_one",
)


print(
    f"Targets with Gaia matches: "
    f"{analysis['gaia_match'].sum():,}"
)

print(
    f"Targets with Gaia QSO flag: "
    f"{analysis['gaia_in_qso_candidates_any'].sum():,}"
)

print(
    f"Targets with MILLiquas match: "
    f"{analysis['milliquas_match'].sum():,}"
)


## 6. Derive diaObject ugrizy mean-flux magnitudes

In [ ]:

alerts = candidate_alerts.copy()

alerts[
    "_props"
] = alerts[
    "alert_properties"
].map(
    parse_properties
)


# ------------------------------------------------------------
# For each member locus and band:
# latest non-null diaObject mean flux.
# ------------------------------------------------------------

locus_flux_rows = []


for locus_id, group in alerts.groupby(
    "locus_id"
):

    group = group.sort_values(
        "mjd"
    )

    row = {
        "locus_id":
            str(
                locus_id
            )
    }


    for band in BANDS:

        key = (
            f"lsst_diaObject_{band}_scienceFluxMean"
        )

        values = []

        for props in group[
            "_props"
        ]:

            value = to_float(
                props.get(
                    key,
                    np.nan,
                )
            )

            if np.isfinite(
                value
            ):
                values.append(
                    value
                )


        row[
            f"diaObject_{band}_scienceFluxMean_locus"
        ] = (
            values[-1]
            if values
            else np.nan
        )


    locus_flux_rows.append(
        row
    )


locus_flux = pd.DataFrame(
    locus_flux_rows
)


# ------------------------------------------------------------
# FoF target: median across member loci.
# ------------------------------------------------------------

target_flux_rows = []


for _, candidate in analysis.iterrows():

    target_id = candidate[
        "target_id"
    ]

    member_ids = [
        str(x)
        for x
        in as_list(
            candidate[
                "member_locus_ids"
            ]
        )
    ]


    member_flux = locus_flux[
        locus_flux[
            "locus_id"
        ].isin(
            member_ids
        )
    ]


    row = {
        "target_id":
            target_id
    }


    for band in BANDS:

        col = (
            f"diaObject_{band}_scienceFluxMean_locus"
        )

        values = pd.to_numeric(
            member_flux[
                col
            ],
            errors="coerce",
        )

        finite = values[
            np.isfinite(
                values
            )
        ]


        flux = (
            finite.median()
            if len(
                finite
            )
            else np.nan
        )


        row[
            f"diaObject_{band}_scienceFluxMean"
        ] = flux

        row[
            f"diaObject_{band}_n_member_flux"
        ] = int(
            len(
                finite
            )
        )

        row[
            f"{band}_mag"
        ] = flux_njy_to_abmag(
            flux
        )


    target_flux_rows.append(
        row
    )


target_flux = pd.DataFrame(
    target_flux_rows
)


analysis = analysis.merge(
    target_flux,
    on="target_id",
    how="left",
    validate="one_to_one",
)


print(
    "diaObject mean-flux magnitudes added:"
)

for band in BANDS:
    print(
        f"  {band}: "
        f"{analysis[f'{band}_mag'].notna().sum():,}"
    )


## 7. Compute median absolute difference-image PSF S/N

In [ ]:

def get_psf_snr_from_props(
    props
):
    flux = to_float(
        props.get(
            "lsst_diaSource_psfFlux",
            np.nan,
        )
    )

    flux_err = to_float(
        props.get(
            "lsst_diaSource_psfFluxErr",
            np.nan,
        )
    )

    if (
        not np.isfinite(
            flux
        )
        or not np.isfinite(
            flux_err
        )
        or flux_err == 0
    ):
        return np.nan

    return abs(
        flux
        / flux_err
    )


alerts[
    "abs_diff_psf_snr"
] = alerts[
    "_props"
].map(
    get_psf_snr_from_props
)


snr_summary = (
    alerts.groupby(
        "target_id"
    )
    .agg(
        median_abs_diff_psf_snr=(
            "abs_diff_psf_snr",
            "median",
        ),
        max_abs_diff_psf_snr=(
            "abs_diff_psf_snr",
            "max",
        ),
    )
    .reset_index()
)


analysis = analysis.merge(
    snr_summary,
    on="target_id",
    how="left",
    validate="one_to_one",
)


## 8. Gaia cleaning — PM and parallax only

In [ ]:

# ============================================================
# KEEP the original PM cut
# ============================================================

analysis[
    "high_pm_flag"
] = (
    analysis[
        "gaia_pm_sig_max"
    ]
    > PM_SIG_CUT
)


# ============================================================
# KEEP the original parallax cut
# ============================================================

analysis[
    "high_parallax_flag"
] = (
    analysis[
        "gaia_parallax_sig_max"
    ]
    > PARALLAX_SIG_CUT
)


# ============================================================
# Gaia color is now a DIAGNOSTIC ONLY.
# It is NOT used in keep_flag.
# ============================================================

analysis[
    "very_red_flag"
] = (
    analysis[
        "gaia_bp_rp_max"
    ]
    >= GAIA_BP_RP_THRESHOLD
)


analysis[
    "gaia_bp_rp_ge_2p5_flag"
] = analysis[
    "very_red_flag"
]


# ============================================================
# CRITICAL CHANGE FROM ORIGINAL v8
#
# OLD:
# keep_flag = ~(
#     high_pm_flag
#     | high_parallax_flag
#     | very_red_flag
# )
#
# NEW:
# ============================================================

analysis[
    "keep_flag"
] = ~(
    analysis[
        "high_pm_flag"
    ]
    | analysis[
        "high_parallax_flag"
    ]
)


def rejection_reason(
    row
):
    reasons = []

    if row[
        "high_pm_flag"
    ]:
        reasons.append(
            "high_pm"
        )

    if row[
        "high_parallax_flag"
    ]:
        reasons.append(
            "high_parallax"
        )

    # Intentionally DO NOT add "very_red".
    return ";".join(
        reasons
    )


analysis[
    "reject_reason"
] = analysis.apply(
    rejection_reason,
    axis=1,
)


print(
    f"Candidates before Gaia cleaning: {len(analysis):,}"
)

print(
    "Rejected by PM cut:              "
    f"{analysis['high_pm_flag'].sum():,}"
)

print(
    "Rejected by parallax cut:        "
    f"{analysis['high_parallax_flag'].sum():,}"
)

print(
    "Gaia BP-RP >= 2.5 (diagnostic): "
    f"{analysis['gaia_bp_rp_ge_2p5_flag'].sum():,}"
)

print(
    "Retained after PM/parallax only: "
    f"{analysis['keep_flag'].sum():,}"
)


## 9. Build full, filtered, and red-Gaia output tables

In [ ]:

# Full table with all flags.
full_list = (
    analysis
    .sort_values(
        "second_pass_score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# Main cleaned sample:
# only PM and parallax cuts are applied.
filtered_list = (
    full_list[
        full_list[
            "keep_flag"
        ]
    ]
    .sort_values(
        [
            "milliquas_match",
            "gaia_in_qso_candidates_any",
            "second_pass_score",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# The targets the user specifically asked for:
# pass PM + parallax cuts AND Gaia BP-RP >= 2.5.
red_gaia_list = (
    filtered_list[
        filtered_list[
            "gaia_bp_rp_ge_2p5_flag"
        ]
    ]
    .sort_values(
        [
            "gaia_bp_rp_max",
            "second_pass_score",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Full candidate sample:                {len(full_list):,}"
)

print(
    f"PM/parallax-cleaned sample:           {len(filtered_list):,}"
)

print(
    f"Cleaned targets with Gaia BP-RP>=2.5: {len(red_gaia_list):,}"
)


if len(
    red_gaia_list
):
    display_columns = [
        col
        for col
        in [
            "target_id",
            "representative_locus_id",
            "member_locus_ids",
            "ra",
            "dec",
            "second_pass_score",
            "gaia_bp_rp_max",
            "gaia_pm_sig_max",
            "gaia_parallax_sig_max",
            "gaia_phot_g_mean_mag_brightest",
            "gaia_in_qso_candidates_any",
            "milliquas_match",
            "u_mag",
            "g_mag",
            "r_mag",
            "i_mag",
            "z_mag",
            "y_mag",
            "median_abs_diff_psf_snr",
        ]
        if col
        in red_gaia_list.columns
    ]

    display(
        red_gaia_list[
            display_columns
        ].head(
            30
        )
    )


## 10. Save results in the new folder

In [ ]:

full_parquet = (
    OUTPUT_DIR
    / "second_pass_candidates_with_flags.parquet"
)

full_csv = (
    OUTPUT_DIR
    / "second_pass_candidates_with_flags.csv"
)

filtered_parquet = (
    OUTPUT_DIR
    / "second_pass_candidates_filtered.parquet"
)

filtered_csv = (
    OUTPUT_DIR
    / "second_pass_candidates_filtered.csv"
)

red_parquet = (
    OUTPUT_DIR
    / "second_pass_candidates_gaia_bp_rp_ge_2p5.parquet"
)

red_csv = (
    OUTPUT_DIR
    / "second_pass_candidates_gaia_bp_rp_ge_2p5.csv"
)


all_output_paths = [
    full_parquet,
    full_csv,
    filtered_parquet,
    filtered_csv,
    red_parquet,
    red_csv,
]


existing = [
    path
    for path
    in all_output_paths
    if path.exists()
]


if (
    existing
    and not OVERWRITE_OUTPUT
):
    raise FileExistsError(
        "Output files already exist and OVERWRITE_OUTPUT=False: "
        f"{existing}"
    )


full_list.to_parquet(
    full_parquet,
    index=False,
)

filtered_list.to_parquet(
    filtered_parquet,
    index=False,
)

red_gaia_list.to_parquet(
    red_parquet,
    index=False,
)


def csv_safe_value(
    value
):
    if isinstance(
        value,
        np.ndarray,
    ):
        value = value.tolist()

    if isinstance(
        value,
        (
            list,
            tuple,
            dict,
        ),
    ):
        return json.dumps(
            value,
            default=str,
            separators=(
                ",",
                ":",
            ),
        )

    return value


def make_csv_safe(
    table
):
    out = table.copy()

    for col in out.columns:

        contains_listlike = any(
            isinstance(
                value,
                (
                    list,
                    tuple,
                    dict,
                    np.ndarray,
                ),
            )
            for value
            in out[
                col
            ]
        )

        if contains_listlike:
            out[
                col
            ] = out[
                col
            ].map(
                csv_safe_value
            )

    return out


make_csv_safe(
    full_list
).to_csv(
    full_csv,
    index=False,
)

make_csv_safe(
    filtered_list
).to_csv(
    filtered_csv,
    index=False,
)

make_csv_safe(
    red_gaia_list
).to_csv(
    red_csv,
    index=False,
)


print(
    "Written:"
)

for path in all_output_paths:
    print(
        " ",
        path,
    )


## 11. Compare the old-color-cut logic with the new selection

In [ ]:

# This is only a comparison; the OLD sample is not saved unless desired.

old_v8_keep_flag = ~(
    analysis[
        "high_pm_flag"
    ]
    | analysis[
        "high_parallax_flag"
    ]
    | analysis[
        "very_red_flag"
    ]
)


n_old = int(
    old_v8_keep_flag.sum()
)

n_new = int(
    analysis[
        "keep_flag"
    ].sum()
)

n_restored = int(
    (
        analysis[
            "keep_flag"
        ]
        & ~old_v8_keep_flag
    ).sum()
)


print(
    f"Would pass original v8 PM+parallax+color cuts: {n_old:,}"
)

print(
    f"Pass new PM+parallax-only cuts:                {n_new:,}"
)

print(
    f"Restored by removing Gaia color cut:           {n_restored:,}"
)


restored = analysis[
    analysis[
        "keep_flag"
    ]
    & ~old_v8_keep_flag
].copy()


# These should be exactly the red-Gaia objects surviving PM/parallax.
check_ids_1 = set(
    restored[
        "target_id"
    ]
)

check_ids_2 = set(
    red_gaia_list[
        "target_id"
    ]
)


if check_ids_1 != check_ids_2:
    raise RuntimeError(
        "The restored-by-color sample does not match "
        "the PM/parallax-cleaned BP-RP>=2.5 sample."
    )


print()
print(
    "Check passed: removing the color cut restores exactly "
    "the PM/parallax-cleaned Gaia BP-RP >= 2.5 targets."
)


## 12. Gaia BP-RP distribution after PM/parallax cleaning

In [ ]:

values = pd.to_numeric(
    filtered_list[
        "gaia_bp_rp_max"
    ],
    errors="coerce",
).dropna()


fig, ax = plt.subplots(
    figsize=(
        4,
        3,
    )
)


if len(
    values
):
    ax.hist(
        values,
        bins=30,
        histtype="step",
    )


ax.axvline(
    GAIA_BP_RP_THRESHOLD,
    color="black",
    ls="--",
)


ax.set_xlabel(
    "Gaia BP-RP"
)

ax.set_ylabel(
    "Candidates"
)


fig.tight_layout()
plt.show()


## 13. Second-Pass score vs Gaia BP-RP

In [ ]:

x = pd.to_numeric(
    filtered_list[
        "second_pass_score"
    ],
    errors="coerce",
)

y = pd.to_numeric(
    filtered_list[
        "gaia_bp_rp_max"
    ],
    errors="coerce",
)

good = (
    x.notna()
    & y.notna()
)


fig, ax = plt.subplots(
    figsize=(
        4,
        3,
    )
)


if good.sum() >= HEXBIN_THRESHOLD:

    hb = ax.hexbin(
        x[
            good
        ],
        y[
            good
        ],
        gridsize=HEXBIN_GRIDSIZE,
        bins="log",
        mincnt=1,
    )

    cbar = fig.colorbar(
        hb,
        ax=ax,
    )

    cbar.set_label(
        "Candidates"
    )

else:

    ax.scatter(
        x[
            good
        ],
        y[
            good
        ],
        s=1,
        alpha=0.1,
    )


ax.axhline(
    GAIA_BP_RP_THRESHOLD,
    color="black",
    ls="--",
)


ax.set_xlabel(
    "Second-Pass score"
)

ax.set_ylabel(
    "Gaia BP-RP"
)


fig.tight_layout()
plt.show()


## 14. Sky distribution of the red-Gaia subset

In [ ]:

def mollweide_ra_dec(
    ra_deg,
    dec_deg,
):
    ra = np.deg2rad(
        (
            np.asarray(
                ra_deg
            )
            + 180.0
        )
        % 360.0
        - 180.0
    )

    # Astronomical convention: RA increases leftward.
    ra = -ra

    dec = np.deg2rad(
        np.asarray(
            dec_deg
        )
    )

    return (
        ra,
        dec,
    )


ra_m, dec_m = mollweide_ra_dec(
    red_gaia_list[
        "ra"
    ],
    red_gaia_list[
        "dec"
    ],
)


fig = plt.figure(
    figsize=(
        6,
        3.5,
    )
)

ax = fig.add_subplot(
    111,
    projection="mollweide",
)

ax.scatter(
    ra_m,
    dec_m,
    s=2,
    alpha=0.3,
)

ax.grid(
    True,
    alpha=0.3,
)

ax.set_title(
    "PM/parallax-cleaned targets with Gaia BP-RP >= 2.5"
)

fig.tight_layout()
plt.show()



## Notes

The key change from the original v8 selection is:

```python
keep_flag = ~(
    high_pm_flag
    | high_parallax_flag
)
```

The color diagnostic is still calculated:

```python
gaia_bp_rp_ge_2p5_flag = gaia_bp_rp_max >= 2.5
```

but it is **not part of `keep_flag`**.

Therefore:

```text
second_pass_candidates_filtered.*
```

contains all candidates that pass the PM and parallax cuts, including red Gaia sources.

And:

```text
second_pass_candidates_gaia_bp_rp_ge_2p5.*
```

contains the subset that passes PM/parallax and has Gaia BP-RP >= 2.5.
